# Concept Shapes v2: Multi-Layer Analysis + Delta Stack

**v1 result**: Final-layer hidden states don't separate Wikipedia categories.
The dominant structure is entropy and prediction mechanics, not topic.

**v2 hypothesis**: Topic information lives in cross-layer structure.
The transformer-as-channel is a composition of morphisms (one per block).
The delta stack δᵢ = hᵢ - hᵢ₋₁ makes that composition explicit.

**Plan**:
1. Collect all 13 layer hidden states per token
2. Per-layer sweep: PCA→KMeans→ARI at each layer (where does topic info peak?)
3. Delta stack: compute deltas, normalise, PCA+AE, UMAP coloured by category
4. Beads on strings in delta-stack latent space

## 0. Setup

In [ ]:
!pip install -q transformers umap-learn scikit-learn matplotlib wikipedia-api pandas 2>/dev/null

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, matplotlib.pyplot as plt, pandas as pd
from sklearn.decomposition import IncrementalPCA, PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, v_measure_score, normalized_mutual_info_score
import gc, os, warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE = '/content/shapes_v2'
os.makedirs(SAVE, exist_ok=True)
print(f'Device: {device}')

## 1. Data Collection: All 13 Layers

Same Wikipedia corpus as v1. Store hidden states from all layers:
embedding (layer 0) + 12 transformer blocks = 13 layers.
Shape: (n_tokens, 13, 768)

In [ ]:
import wikipediaapi

CATEGORIES = {
    'physics': [
        'Quantum mechanics', 'General relativity', 'Thermodynamics',
        'Electromagnetism', 'Particle physics', 'Classical mechanics',
        'Quantum field theory', 'Special relativity', 'Nuclear physics',
        'Condensed matter physics', 'Astrophysics', 'Optics',
        'Fluid dynamics', 'Plasma physics', 'Solid-state physics',
        'Statistical mechanics', 'Atomic physics', 'Biophysics',
        'Geophysics', 'Acoustics', 'Photonics', 'Superconductivity',
        'Quantum entanglement', 'Dark matter', 'Higgs boson',
        'Wave–particle duality', 'Schrödinger equation', 'Standard Model',
        'String theory', 'Quantum computing'
    ],
    'cooking': [
        'Sautéing', 'Baking', 'Fermentation in food processing',
        'Sous vide', 'Pastry', 'Bread', 'Curry', 'Sushi',
        'French cuisine', 'Italian cuisine', 'Chinese cuisine',
        'Spice', 'Chocolate', 'Cheese', 'Wine', 'Olive oil',
        'Barbecue', 'Roasting', 'Braising', 'Pickling',
        'Sourdough', 'Ramen', 'Dim sum', 'Miso',
        'Butter', 'Flour', 'Yeast', 'Caramelization',
        'Maillard reaction', 'Smoking (cooking)'
    ],
    'sport': [
        'Association football', 'Basketball', 'Tennis',
        'Cricket', 'Rugby union', 'Baseball', 'Ice hockey',
        'Golf', 'Swimming (sport)', 'Athletics (sport)',
        'Boxing', 'Cycling', 'Volleyball', 'Table tennis',
        'Badminton', 'Fencing', 'Rowing (sport)', 'Sailing',
        'Surfing', 'Skiing', 'Marathon', 'Triathlon',
        'Judo', 'Karate', 'Archery', 'Weightlifting',
        'American football', 'Handball', 'Water polo', 'Lacrosse'
    ],
    'law': [
        'Contract law', 'Criminal law', 'Constitutional law',
        'International law', 'Tort', 'Property law', 'Family law',
        'Corporate law', 'Tax law', 'Labour law',
        'Human rights', 'Habeas corpus', 'Due process',
        'Jury trial', 'Common law', 'Civil law (legal system)',
        'Supreme Court of the United States', 'Magna Carta',
        'Rule of law', 'Intellectual property', 'Patent law',
        'Copyright law', 'Immigration law', 'Environmental law',
        'Admiralty law', 'Antitrust law', 'Bankruptcy',
        'Evidence (law)', 'Legal precedent', 'Statute'
    ],
    'music': [
        'Music theory', 'Musical notation', 'Harmony',
        'Melody', 'Rhythm', 'Counterpoint', 'Sonata form',
        'Symphony', 'Opera', 'Jazz', 'Blues',
        'Electronic music', 'Hip hop music', 'Rock music',
        'Classical music', 'Folk music', 'Reggae',
        'Musical instrument', 'Piano', 'Violin', 'Guitar',
        'Drum kit', 'Synthesizer', 'Orchestra', 'Choir',
        'Music production', 'Sound recording and reproduction',
        'Tempo', 'Key (music)', 'Scale (music)'
    ],
    'medicine': [
        'Cardiology', 'Neurology', 'Oncology', 'Immunology',
        'Pharmacology', 'Surgery', 'Pathology', 'Epidemiology',
        'Radiology', 'Psychiatry', 'Dermatology', 'Pediatrics',
        'Anesthesiology', 'Gastroenterology', 'Endocrinology',
        'Nephrology', 'Pulmonology', 'Rheumatology',
        'Hematology', 'Infectious disease', 'Vaccine',
        'Antibiotic', 'Clinical trial', 'Medical imaging',
        'Blood pressure', 'Diabetes', 'Cancer',
        'Cardiovascular disease', 'Stroke', 'Alzheimer\'s disease'
    ]
}

SEEDS_PER_CAT = 30

wiki = wikipediaapi.Wikipedia(user_agent='ConceptShapes/2.0 (research)', language='en')
articles = {}
for cat, titles in CATEGORIES.items():
    articles[cat] = []
    for title in titles:
        if len(articles[cat]) >= SEEDS_PER_CAT: break
        page = wiki.page(title)
        if page.exists() and len(page.text) > 500:
            articles[cat].append((title, page.text))
    print(f'{cat}: {len(articles[cat])} articles')
print(f'Total: {sum(len(v) for v in articles.values())}')

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
torch.manual_seed(42)

PREFIX_LEN = 128; CONT_LEN = 128; D_MODEL = 768; N_LAYERS = 13

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2').to(device); model.eval()

prefixes = []
for cat, arts in articles.items():
    for title, text in arts:
        toks = tokenizer.encode(text, max_length=PREFIX_LEN + CONT_LEN, truncation=True)
        if len(toks) >= PREFIX_LEN + 10:
            prefixes.append((cat, title, toks[:PREFIX_LEN]))

print(f'{len(prefixes)} usable prefixes')
for cat in CATEGORIES:
    print(f'  {cat}: {sum(1 for c,_,_ in prefixes if c==cat)}')

In [ ]:
# Collect ALL 13 layer hidden states during free generation
# Store as (n_total, 13, 768) in fp16

NUM_SEQ = len(prefixes)
n_total = NUM_SEQ * CONT_LEN

# Preallocate on disk
hpath = f'{SAVE}/hidden_all_layers.bin'
hs = torch.HalfStorage.from_file(hpath, shared=True, size=n_total * N_LAYERS * D_MODEL)
hidden_all = torch.HalfTensor(hs).reshape(n_total, N_LAYERS, D_MODEL)

all_meta = []; wi = 0

print(f'Collecting {n_total} vectors × {N_LAYERS} layers from {NUM_SEQ} sequences...')
with torch.no_grad():
    for si, (cat, title, prefix) in enumerate(prefixes):
        ids = torch.tensor([prefix], device=device)

        for st in range(CONT_LEN):
            out = model(ids, output_hidden_states=True)
            # out.hidden_states is tuple of (batch, seq_len, 768), length 13
            # We want the last token position from each layer
            for li in range(N_LAYERS):
                hidden_all[wi, li] = out.hidden_states[li][0, -1, :].cpu().half()

            rl = out.logits[0, -1, :]
            lp = F.log_softmax(rl, dim=-1)
            ent = -(torch.exp(lp) * lp).sum().item()
            nt = torch.multinomial(F.softmax(rl, dim=-1), 1)

            all_meta.append({
                'seq': si, 'step': st, 'category': cat, 'title': title,
                'token_id': nt.item(),
                'token_str': tokenizer.decode(nt.item()),
                'entropy': ent
            })
            wi += 1

            ids = torch.cat([ids, nt.unsqueeze(0)], dim=-1)
            if ids.shape[1] > 1024: ids = ids[:, -1024:]
            del out, rl, lp

        if (si+1) % 20 == 0: print(f'  {si+1}/{NUM_SEQ}')

hidden_all = hidden_all[:wi]
torch.save(all_meta, f'{SAVE}/meta.pt')
del model; torch.cuda.empty_cache(); gc.collect()
print(f'Done: {wi} vectors, shape {hidden_all.shape}')
print(f'Storage: {hidden_all.nelement()*2/1e6:.0f} MB')

## 2. Per-Layer Sweep: Where Does Topic Info Live?

For each of the 13 layers independently: PCA → KMeans → ARI.
No AE (v1 showed it adds nothing over PCA). This gives the
**concept localisation curve**: ARI as a function of depth.

In [ ]:
cats = np.array([m['category'] for m in all_meta])
cat_names = sorted(CATEGORIES.keys())
cat_to_idx = {c: i for i, c in enumerate(cat_names)}
cat_idx = np.array([cat_to_idx[c] for c in cats])
n_clusters = len(cat_names)

pca_dim = 32  # project to this many dims before clustering

layer_results = []
for li in range(N_LAYERS):
    h = hidden_all[:, li, :].float().numpy()

    # PCA
    pca = PCA(n_components=pca_dim, random_state=42)
    h_pca = pca.fit_transform(h)
    var95 = int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.95)) + 1

    # KMeans on PCA projection
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cl = km.fit_predict(h_pca)

    ari = adjusted_rand_score(cat_idx, cl)
    nmi = normalized_mutual_info_score(cat_idx, cl)
    vms = v_measure_score(cat_idx, cl)

    # Also: per-layer norm stats
    norms = np.linalg.norm(h, axis=1)

    layer_results.append({
        'layer': li, 'ari': ari, 'nmi': nmi, 'vms': vms,
        'var95': var95, 'top1_var': pca.explained_variance_ratio_[0],
        'norm_mean': norms.mean(), 'norm_std': norms.std()
    })
    print(f'Layer {li:2d}: ARI={ari:.4f}  NMI={nmi:.4f}  95%@{var95}  top1_var={pca.explained_variance_ratio_[0]:.3f}  norm={norms.mean():.1f}±{norms.std():.1f}')

lr_df = pd.DataFrame(layer_results)
del h, h_pca; gc.collect()

In [ ]:
# === CONCEPT LOCALISATION CURVE ===
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

ax = axes[0]
ax.plot(lr_df['layer'], lr_df['ari'], 'bo-', linewidth=2, markersize=8, label='ARI')
ax.plot(lr_df['layer'], lr_df['nmi'], 'rs--', linewidth=1.5, markersize=6, label='NMI')
ax.set_xlabel('Layer'); ax.set_ylabel('Score')
ax.set_title('Concept Localisation Curve', fontsize=12)
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xticks(range(N_LAYERS))

ax = axes[1]
ax.plot(lr_df['layer'], lr_df['top1_var'], 'go-', linewidth=2, markersize=8)
ax.set_xlabel('Layer'); ax.set_ylabel('Variance ratio')
ax.set_title('Top-1 PCA Variance by Layer'); ax.grid(True, alpha=0.3)
ax.set_xticks(range(N_LAYERS))

ax = axes[2]
ax.plot(lr_df['layer'], lr_df['var95'], 'mo-', linewidth=2, markersize=8)
ax.set_xlabel('Layer'); ax.set_ylabel('Components for 95%')
ax.set_title('Intrinsic Dimensionality by Layer'); ax.grid(True, alpha=0.3)
ax.set_xticks(range(N_LAYERS))

ax = axes[3]
ax.errorbar(lr_df['layer'], lr_df['norm_mean'], yerr=lr_df['norm_std'],
            fmt='ko-', linewidth=2, markersize=6, capsize=3)
ax.set_xlabel('Layer'); ax.set_ylabel('L2 Norm')
ax.set_title('Hidden State Norms by Layer'); ax.grid(True, alpha=0.3)
ax.set_xticks(range(N_LAYERS))

plt.suptitle('Per-Layer Analysis: Where Does Topic Information Live?', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{SAVE}/layer_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

peak_layer = lr_df.loc[lr_df['ari'].idxmax(), 'layer']
print(f'\nPeak ARI at layer {peak_layer}: {lr_df["ari"].max():.4f}')

In [ ]:
# UMAP at the peak layer vs final layer — visual comparison
import umap

colors = plt.cm.tab10(np.linspace(0, 1, len(cat_names)))
peak_li = int(peak_layer)

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, li, title in zip(axes, [0, peak_li, 12],
                         [f'Layer 0 (embedding)', f'Layer {peak_li} (peak ARI)', 'Layer 12 (final)']):
    h = hidden_all[:, li, :].float().numpy()
    pca = PCA(n_components=32, random_state=42)
    h_pca = pca.fit_transform(h)
    emb = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=42).fit_transform(h_pca)
    for i, cat in enumerate(cat_names):
        mask = cats == cat
        ax.scatter(emb[mask, 0], emb[mask, 1], c=[colors[i]], s=2, alpha=0.3, label=cat)
    ari_here = lr_df[lr_df['layer']==li]['ari'].values[0]
    ax.set_title(f'{title}\nARI={ari_here:.4f}', fontsize=11)
    ax.legend(markerscale=5, fontsize=8); ax.set_aspect('equal')

plt.suptitle('Category Separation Across Layers', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{SAVE}/umap_layer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Delta Stack

δ₀ = h₀, δᵢ = hᵢ - hᵢ₋₁ for i=1..12.
Stack into (n_tokens, 12×768) or (n_tokens, 13×768).

This is a lossless linear transform of the naive stack but decorrelates
the residual stream redundancy. Each 768-d block is what a transformer
block **added** — the morphism content of that layer.

In [ ]:
# Compute deltas
print('Computing delta stack...')
n = len(hidden_all)

# δ₀ = h₀, δᵢ = hᵢ - hᵢ₋₁
deltas = torch.zeros(n, N_LAYERS, D_MODEL, dtype=torch.float32)
deltas[:, 0, :] = hidden_all[:, 0, :].float()
for li in range(1, N_LAYERS):
    deltas[:, li, :] = hidden_all[:, li, :].float() - hidden_all[:, li-1, :].float()

# Check norms per layer
print('\nDelta norms per layer:')
for li in range(N_LAYERS):
    norms = deltas[:, li, :].norm(dim=-1)
    print(f'  δ_{li:2d}: mean={norms.mean():.2f}  std={norms.std():.2f}')

In [ ]:
# Build two versions: raw stack and normalised stack

# Raw: just flatten
delta_raw = deltas.reshape(n, N_LAYERS * D_MODEL).numpy()
print(f'Raw delta stack: {delta_raw.shape}')

# Normalised: each 768-d block scaled to unit norm
deltas_normed = deltas.clone()
for li in range(N_LAYERS):
    norms = deltas_normed[:, li, :].norm(dim=-1, keepdim=True).clamp(min=1e-8)
    deltas_normed[:, li, :] /= norms
delta_normed = deltas_normed.reshape(n, N_LAYERS * D_MODEL).numpy()
print(f'Normed delta stack: {delta_normed.shape}')

del deltas_normed; gc.collect()

In [ ]:
# PCA on both versions — what's the effective dimensionality?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, name in zip(axes, [delta_raw, delta_normed], ['Raw deltas', 'Normed deltas']):
    # Use incremental PCA for memory
    ipca = IncrementalPCA(n_components=128, batch_size=2000)
    for i in range(0, len(data), 2000):
        ipca.partial_fit(data[i:i+2000])
    ev = ipca.explained_variance_ratio_
    cs = np.cumsum(ev)
    n95 = int(np.searchsorted(cs, 0.95)) + 1
    n99 = int(np.searchsorted(cs, 0.99)) + 1
    ax.plot(cs, 'b-', linewidth=2)
    ax.axhline(0.95, c='r', ls='--', alpha=0.5, label=f'95% @ {n95}')
    ax.axhline(0.99, c='orange', ls='--', alpha=0.5, label=f'99% @ {n99}')
    ax.set_xlabel('Components'); ax.set_ylabel('Cumulative variance')
    ax.set_title(f'{name}: 95%@{n95}, 99%@{n99}'); ax.legend(); ax.grid(True, alpha=0.3)
    print(f'{name}: 95%@{n95}, 99%@{n99}, top-5 var: {ev[:5].round(4)}')

plt.tight_layout(); plt.savefig(f'{SAVE}/delta_pca_spectrum.png', dpi=150); plt.show()

In [ ]:
# KMeans on delta stack PCA — does cross-layer structure separate categories?
# Test both raw and normed

results_delta = {}
for data, name in [(delta_raw, 'raw'), (delta_normed, 'normed')]:
    pca = IncrementalPCA(n_components=64, batch_size=2000)
    for i in range(0, len(data), 2000): pca.partial_fit(data[i:i+2000])
    proj = np.empty((len(data), 64), dtype=np.float32)
    for i in range(0, len(data), 2000):
        chunk = data[i:i+2000]
        proj[i:i+len(chunk)] = pca.transform(chunk)

    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cl = km.fit_predict(proj)
    ari = adjusted_rand_score(cat_idx, cl)
    nmi = normalized_mutual_info_score(cat_idx, cl)
    results_delta[name] = {'ari': ari, 'nmi': nmi, 'pca': pca, 'proj': proj}
    print(f'Delta stack ({name}): ARI={ari:.4f}, NMI={nmi:.4f}')

# Compare with best single-layer result
print(f'\nBest single layer (layer {peak_layer}): ARI={lr_df["ari"].max():.4f}')
print(f'Final layer (layer 12): ARI={lr_df[lr_df["layer"]==12]["ari"].values[0]:.4f}')

In [ ]:
# AE on the better delta variant
best_delta = 'normed' if results_delta['normed']['ari'] >= results_delta['raw']['ari'] else 'raw'
print(f'Using {best_delta} deltas for AE')
delta_data = delta_normed if best_delta == 'normed' else delta_raw

# Two-stage: PCA to 128, then AE
print('PCA to 128...')
pca_pre = IncrementalPCA(n_components=128, batch_size=2000)
for i in range(0, len(delta_data), 2000): pca_pre.partial_fit(delta_data[i:i+2000])
delta_pca = np.empty((len(delta_data), 128), dtype=np.float32)
for i in range(0, len(delta_data), 2000):
    chunk = delta_data[i:i+2000]
    delta_pca[i:i+len(chunk)] = pca_pre.transform(chunk)

delta_pca_t = torch.tensor(delta_pca)

class AE(nn.Module):
    def __init__(s, ind, hid, lat):
        super().__init__()
        s.enc = nn.Sequential(nn.Linear(ind, hid), nn.ReLU(), nn.Linear(hid, lat))
        s.dec = nn.Sequential(nn.Linear(lat, hid), nn.ReLU(), nn.Linear(hid, ind))
    def encode(s, x): return s.enc(x)
    def forward(s, x): z = s.enc(x); return s.dec(z), z

def train_ae(data, k, ind=128, hid=256, ep=50, bs=512, lr=1e-3):
    n = len(data); perm = torch.randperm(n); nt = int(0.8*n)
    tr, te = data[perm[:nt]], data[perm[nt:]]
    m = AE(ind, hid, k).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, ep)
    ld = DataLoader(TensorDataset(tr), batch_size=bs, shuffle=True)
    trh, teh = [], []
    for e in range(ep):
        m.train(); tot = 0
        for (b,) in ld:
            b = b.to(device); r, z = m(b)
            l = F.mse_loss(r, b); opt.zero_grad(); l.backward(); opt.step()
            tot += l.item()*len(b)
        trh.append(tot/nt); sch.step()
        if (e+1)%10==0 or e==ep-1:
            m.eval()
            with torch.no_grad(): r2,_=m(te.to(device)); teh.append((e,F.mse_loss(r2,te.to(device)).item()))
    return m, trh, teh

sweep = {}
for k in [16, 32, 64]:
    print(f'k={k}...', end=' ')
    m, th, teh = train_ae(delta_pca_t, k)
    ft, fte = th[-1], teh[-1][1]
    r = fte/ft if ft > 1e-15 else float('inf')
    sweep[k] = {'ft': ft, 'fte': fte, 'model': m}
    print(f'train={ft:.3e} test={fte:.3e} ratio={r:.2f}x')

In [ ]:
# Select K, encode, UMAP
K = 32  # ← adjust based on sweep
ae = sweep[K]['model']; ae.eval()

with torch.no_grad():
    latents = torch.cat([ae.encode(delta_pca_t[i:i+500].to(device)).cpu()
                         for i in range(0, len(delta_pca_t), 500)])
print(f'Latent shape: {latents.shape}')

# Cluster quality
km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cl = km.fit_predict(latents.numpy())
ari_ae = adjusted_rand_score(cat_idx, cl)
nmi_ae = normalized_mutual_info_score(cat_idx, cl)
print(f'Delta AE k={K}: ARI={ari_ae:.4f}, NMI={nmi_ae:.4f}')
print(f'vs PCA-only: ARI={results_delta[best_delta]["ari"]:.4f}')

In [ ]:
# UMAP on delta-stack latent
print('UMAP on delta-stack AE latent...')
umap_delta = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=42).fit_transform(latents.numpy())

ents = np.array([m['entropy'] for m in all_meta])
steps = np.array([m['step'] for m in all_meta])

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

ax = axes[0]
for i, cat in enumerate(cat_names):
    mask = cats == cat
    ax.scatter(umap_delta[mask, 0], umap_delta[mask, 1], c=[colors[i]], s=2, alpha=0.3, label=cat)
ax.legend(markerscale=5, fontsize=9)
ax.set_title(f'Delta Stack AE k={K} — Category\nARI={ari_ae:.4f}', fontsize=12)
ax.set_aspect('equal')

ax = axes[1]
sc = ax.scatter(umap_delta[:, 0], umap_delta[:, 1], c=ents, cmap='plasma', s=2, alpha=0.3)
plt.colorbar(sc, ax=ax, shrink=0.7); ax.set_title('Entropy', fontsize=12); ax.set_aspect('equal')

ax = axes[2]
sc = ax.scatter(umap_delta[:, 0], umap_delta[:, 1], c=steps, cmap='viridis', s=2, alpha=0.3)
plt.colorbar(sc, ax=ax, shrink=0.7); ax.set_title('Position', fontsize=12); ax.set_aspect('equal')

plt.suptitle('Delta Stack Latent Space — Does Cross-Layer Structure Separate Categories?', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{SAVE}/delta_umap_categories.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Beads on Strings: Delta-Stack Trajectories

If the delta-stack latent separates categories, trajectories here
live in a space where axes reflect *computational contributions*,
not just final-layer state.

In [ ]:
import random
random.seed(42)

seq_ids_by_cat = {}
for cat in cat_names:
    cat_seqs = sorted(set(m['seq'] for m in all_meta if m['category'] == cat))
    seq_ids_by_cat[cat] = random.sample(cat_seqs, min(3, len(cat_seqs)))

# Within-category trajectory panels
fig, axes = plt.subplots(2, 3, figsize=(20, 13))

for ax, (cat_i, cat) in zip(axes.flat, enumerate(cat_names)):
    ax.scatter(umap_delta[:, 0], umap_delta[:, 1], c='lightgrey', s=0.3, alpha=0.05)

    cat_seqs = sorted(set(m['seq'] for m in all_meta if m['category'] == cat))
    for seq_id in cat_seqs:
        idxs = [j for j, m in enumerate(all_meta) if m['seq'] == seq_id]
        if len(idxs) < 2: continue
        traj = umap_delta[idxs]
        ax.plot(traj[:, 0], traj[:, 1], '-', color=colors[cat_i], alpha=0.4, linewidth=0.8)
        ax.scatter(traj[0, 0], traj[0, 1], c=[colors[cat_i]], s=20, marker='o',
                   edgecolors='black', linewidths=0.5, zorder=5)

    ax.set_title(f'{cat} ({len(cat_seqs)} sequences)', fontsize=12, color=colors[cat_i])
    ax.set_aspect('equal')

plt.suptitle('Delta-Stack Trajectories by Category', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{SAVE}/delta_trajectories_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Trajectory dynamics in delta-stack latent space
traj_stats = []
for si in range(NUM_SEQ):
    idxs = [j for j, m in enumerate(all_meta) if m['seq'] == si]
    if len(idxs) < 3: continue
    z = latents[idxs].numpy()
    cat = all_meta[idxs[0]]['category']

    diffs = np.diff(z, axis=0)
    velocities = np.linalg.norm(diffs, axis=1)
    if len(diffs) > 1:
        accels = np.linalg.norm(np.diff(diffs, axis=0), axis=1)
    else:
        accels = np.array([0.0])

    path_len = velocities.sum()
    displacement = np.linalg.norm(z[-1] - z[0])
    tortuosity = path_len / (displacement + 1e-8)

    traj_stats.append({
        'seq': si, 'category': cat,
        'mean_velocity': velocities.mean(),
        'std_velocity': velocities.std(),
        'mean_accel': accels.mean(),
        'path_length': path_len,
        'displacement': displacement,
        'tortuosity': tortuosity
    })

df = pd.DataFrame(traj_stats)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, metric in zip(axes, ['mean_velocity', 'tortuosity', 'displacement', 'path_length']):
    data_by_cat = [df[df['category']==cat][metric].values for cat in cat_names]
    bp = ax.boxplot(data_by_cat, labels=cat_names, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors): patch.set_facecolor(color)
    ax.set_title(metric.replace('_', ' ').title(), fontsize=11)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Trajectory Dynamics in Delta-Stack Latent Space', fontsize=13)
plt.tight_layout(); plt.savefig(f'{SAVE}/delta_trajectory_stats.png', dpi=150); plt.show()

print(df.groupby('category')[['mean_velocity', 'tortuosity', 'displacement', 'path_length']].mean().round(3))

In [ ]:
# UMAP robustness on delta stack
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (nn, md) in zip(axes, [(15, 0.1), (30, 0.3), (50, 0.5)]):
    emb = umap.UMAP(n_neighbors=nn, min_dist=md, random_state=42).fit_transform(latents.numpy())
    for i, cat in enumerate(cat_names):
        mask = cats == cat
        ax.scatter(emb[mask, 0], emb[mask, 1], c=[colors[i]], s=2, alpha=0.3, label=cat)
    ax.set_title(f'nn={nn}, min_dist={md}'); ax.set_aspect('equal')
    if ax == axes[0]: ax.legend(markerscale=4, fontsize=8)
plt.suptitle('Delta Stack UMAP Robustness', fontsize=13)
plt.tight_layout(); plt.savefig(f'{SAVE}/delta_umap_robustness.png', dpi=150); plt.show()

## 5. Summary & Comparison

In [ ]:
print('='*60)
print('CONCEPT SHAPES v2 — SUMMARY')
print('='*60)
print(f'\nCorpus: {NUM_SEQ} sequences, {CONT_LEN} tokens each, {len(cat_names)} categories')
print(f'\n--- Per-Layer Concept Localisation ---')
print(lr_df[['layer', 'ari', 'nmi', 'top1_var', 'norm_mean']].to_string(index=False))
print(f'\nPeak: layer {peak_layer}, ARI={lr_df["ari"].max():.4f}')
print(f'\n--- Delta Stack ---')
for name in ['raw', 'normed']:
    r = results_delta[name]
    print(f'  {name}: ARI={r["ari"]:.4f}, NMI={r["nmi"]:.4f}')
print(f'  AE k={K}: ARI={ari_ae:.4f}, NMI={nmi_ae:.4f}')
print(f'\n--- Comparison ---')
print(f'  Final layer only (v1):  ARI ~ 0  (no separation)')
print(f'  Best single layer:      ARI = {lr_df["ari"].max():.4f} (layer {peak_layer})')
print(f'  Delta stack ({best_delta}):     ARI = {results_delta[best_delta]["ari"]:.4f}')
print(f'  Delta stack AE k={K}:   ARI = {ari_ae:.4f}')